# 은비 개인 통합 정리 노트북 (완성본)

- **프로젝트명**: 성남시 젠트리피케이션 분석
- **담당자명**: 은비
- **담당 파트**: 상업용 부동산 거래량 / 인구 / 신규 기업 데이터 전처리 및 상업용 부동산 거래량 / 신규 기업 / 카드(매출금) 데이터 EDA
- **작성일**: 2026-04-29
- **본 노트북 목적**: 전처리 및 EDA 내용을 팀원에게 공유하기 위해 기록을 남긴다.
- **최종 산출물 한 줄 요약**: 거래량 데이터를 법정동·월 단위 clean 파일로, 인구 / 기업 / 매출 데이터를 행정동·월 단위 clean 파일로 만들고 거래량 데이터에 대한 법정동·분기 단위 EDA 결과와 기업 / 매출 데이터에 대한 행정동·분기 단위 EDA 결과를 정리한다.

## 1. 작업 개요

- **내가 맡은 데이터/업무**
  - (1) 국토교통부 실거래가공개시스템 상업/업무용 부동산 매매거래 데이터 → 거래량 변수 추출
  - (2) 행정동 단위 인구/세대수 데이터 → 배후 수요 변수 추출
  - (3) 행정동 단위 신규 법인(기업) 데이터 → 상권 변화 신호 변수 추출
  - (4) 카드 매출 데이터 → 행정동별 매출 흐름 분석
- **왜 필요한가**: 성남시(분당구·수정구·중원구) 내 젠트리피케이션 위험 지역을 식별하기 위해, 부동산 가치 변화 / 배후 수요 / 상권 활동을 동일 단위에서 비교해야 한다.
- **최종적으로 남길 결과물**
  - clean 파일: `실거래가_2차_전처리.csv`, `population_total.csv` 정리본, `new_corp_total.csv` 정리본
  - 수준 변수 / 변화율 변수 구분표
  - 최종 변수 연결표
  - EDA 핵심 결과 요약(분기별 거래량 상위 동, 신규기업 증가율, 행정동별 매출 top, 도메인 조사 위험 수준 표)

## 2. 파일 분류표

| 파일명 | 구분 | 현재 역할 | 조치 계획 | 비고 |
|---|---|---|---|---|
| new_거래량_2차 전처리.ipynb | **최종본** | 거래량 데이터 결측/이상치 점검 | 본 노트북에 핵심 내용 통합 완료 | 거래량 파트 원본 |
| 인구_기업_전처리.ipynb | **최종본** | 인구/기업 데이터 점검 | 본 노트북에 핵심 내용 통합 완료 | 인구·기업 파트 원본 |
| EDA.ipynb | **최종본** | 거래량/신규기업/매출 EDA | 본 노트북 8장으로 결과 요약 | EDA 원본 |

## 3. 입력 파일 정리 + 단위 확정 칸

| 파일명 | 데이터 종류 | 시간 단위 | 공간 단위 | 사용 여부 | 비고 |
|---|---|---|---|---|---|
| transactions_total.csv | 부동산 매매 거래 | 월 (계약연월) | 시군구 + 법정동 | 사용 | SQL 1차 전처리 완료 (3,060행 × 9컬럼) |
| population_total.csv | 인구 / 세대수 | 월 (날짜) | 행정동 + 행정동 코드 | 사용 | SQL 1차 전처리 완료 (1,800행 × 6컬럼) |
| new_corp_total.csv | 신규 법인 수 (업종별) | 월 (date) | 행정동 + dong_code | 사용 | SQL 1차 전처리 완료(3,367행 × 10컬럼) |
| 전처리완료_카드(매출).csv | 카드 매출/거래 건수 | 일 (ta_ymd) | 행정동(admi_cty_no) | 사용 | 근수님 제공(1·2차 전처리 담당)|
| seongnam_dong_master.csv | 행정동 코드-이름 매핑 | - | 행정동 | 사용 | 매출 → 동명 매핑용, 길래 튜터님 제공 | 

### 시간 단위 확정
- **최종 시간 단위: 분기별** 

### 공간 단위 확정
- **최종 공간 단위: 행정동** — 거래량 데이터는 법정동 기준이므로, 분석 단계에서 법정동↔행정동 매핑을 별도로 확인해야 한다.
- 분석 대상 구: 성남시 **분당구 / 수정구 / 중원구** 3개 구.

## 4. 환경 설정 및 데이터 로드

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import duckdb
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.0f}'.format)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

FILE_PATH = Path('C:/python_course/final_project/eunbi')
print(f'FILE_PATH: {FILE_PATH}')

FILE_PATH: C:\python_course\final_project\eunbi


In [2]:
source_files = {
    'transactions': (FILE_PATH / 'transactions_total.csv', 'cp949'),
    'population':   (FILE_PATH / 'population_total.csv',   'euc-kr'),
    'new_corp':     (FILE_PATH / 'new_corp_total.csv',     'utf-8'),
    'sales':        (FILE_PATH / '전처리완료_카드(매출).csv', 'utf-8'),
    'dong_master':  (FILE_PATH / 'seongnam_dong_master.csv', 'cp949'),
}

loaded = {}
for name, (path, enc) in source_files.items():
    if path.exists():
        df = pd.read_csv(path, encoding=enc)
        df.columns = [str(c).strip() for c in df.columns]
        loaded[name] = df
        print(f'[{name}] 로드 완료 -> {df.shape}')
    else:
        print(f'[{name}] 파일 없음 -> {path}')

[transactions] 로드 완료 -> (3060, 9)
[population] 로드 완료 -> (1800, 6)
[new_corp] 로드 완료 -> (3367, 10)
[sales] 로드 완료 -> (87263128, 12)
[dong_master] 로드 완료 -> (50, 10)


## 5. 데이터 기본 점검 결과 요약

각 데이터셋에 대해 `info()`, `describe()`, 결측치, 중복행을 점검한 결과를 표로 정리한다.

### 5-1. 거래량 (`transactions_total.csv`)
- **shape**: 3,060행 × 9컬럼
- **컬럼**: 시군구, 법정동, 유형, 용도지역, 전용/연면적(㎡), 건축물주용도, 거래금액(만원), 층, 계약연월
- **중복행**: 0건 (MySQL 1차 전처리에서 DISTINCT 처리)
- **결측치**
  - 시군구, 법정동, 유형, 용도지역, 전용/연면적, 건축물주용도, 거래금액, 계약연월: 0개 (0.0%)
  - 층: **1,408개 (46.01%)**
- **유형 분포**: 집합 2,871건 / 일반 189건
- **용도지역 분포 (상위)**: 중심상업 679 / 준주거 587 / 일반상업 584 / 근린상업 484 / 제3종일반주거 332 …
- **건축물주용도 분포**: 제2종근린생활 1,075 / 제1종근린생활 883 / 업무 456 / 판매 305 / 기타 204 / 교육연구 126 / 숙박 11

### 5-2. 인구 (`population_total.csv`)
- **shape**: 1,800행 × 6컬럼
- **컬럼**: 행정구역, 행정동 코드, 날짜, 총인구수, 세대수, 세대당_인구
- **중복행**: 0건
- **결측치**: 모든 컬럼 0개
- **타입 이슈**: `총인구수`, `세대수`가 콤마 포함 문자열로 저장됨 → 정수 변환 필요
- **describe (변환 후)**
  - 총인구수: min 2,911 / 평균 18,310 / max 46,396
  - 세대수: min 1,523 / 평균 8,196 / max 18,849
  - 세대당_인구: min 1.54 / 평균 2.23 / max 3.28

### 5-3. 신규 기업 (`new_corp_total.csv`)
- **shape**: 3,367행 × 10컬럼
- **컬럼**: date, sido_nm, sigun_nm, admi_nm, dong_code, induty_pri_cd, induty_pri_nm, induty_med_cd, induty_med_nm, ncr_crp_comp_cn
- **중복행**: 0건 (`date + dong_code + induty_pri_nm + induty_med_nm` 기준)
- **결측치**: 모든 컬럼 0개
- **describe**: ncr_crp_comp_cn min 1 / 평균 1.28 / max 12 (행정동·월·업종별 신규 법인 수)

## 6. 문제 데이터 정리 표 + 처리 기준 문서화

### 6-1. 문제 데이터 정리 표

| 데이터셋 | 컬럼명 | 문제 유형 | 처리 방식 | 이유 | 확인 상태 |
|---|---|---|---|---|---|
| 거래량 | 층 | 결측치 46.01% (1,408건) | (일반) NaN → `whole_building`, (집합) NaN → `unknown` | 일반 유형은 통건물 매매로 층 구분 자체가 없음. 집합 유형 결측은 지하/구분 모호 → 거래금액 이상치 판단용 보조지표 수준이라 라벨로 대체. 튜터 답변 기준. | **완료** |
| 거래량 | 거래금액(만원) | 최솟값 200만원(2백만원) ~ 최댓값 198,204,140만원(약 1.98조원). 상업용 200만원은 비정상적으로 작음 | 상·하위 10% 중에서도 IQR 기반 극단값 추출(상위: Q3+1.5·IQR, 하위: Q1−0.5·IQR) | 보수적으로 봐도 1,000~1,500만원이 합리적 최소선. 단순 절대값 컷보다 분포 기반 검토. | **완료**(추출), 제거 여부 추가 검토 |
| 거래량 | 전용/연면적(㎡) | 최솟값 4㎡(약 1.21평) ~ 최댓값 197,237㎡(약 5.97만평). 면적이 작을수록 금액도 비정상적으로 낮음 | 거래금액-면적 상관관계 + 분포 보면서 이상치 컷 기준 결정 예정 | 단일 임계값으로 자르기 어려움. 두 변수 함께 봐야 함. | **검토 중** |
| 인구 | 총인구수, 세대수 | 콤마 포함 문자열 → 숫자 연산 불가 | `str.replace(',', '').astype(int)` | 집계/시각화 위해 정수 타입 필요. | **완료** |
| 매출 | ta_ymd | 일 단위 데이터 | 분기 단위로 집계(`STRFTIME('%Y-%m')`) | 다른 데이터셋과 단위 통일. | **완료**(EDA 단계) |
| 매출 | admi_cty_no | 8자리 행정동 코드 | `seongnam_dong_master.csv`의 `dong_code_8`과 LEFT JOIN | 동명 매핑 후 해석 가능. | **완료** |

### 6-2. 처리 기준 문서화
- **원본 유지 여부**: 원본 csv는 수정하지 않고, 분석용 clean 파일을 별도로 저장한다.
- **집계 기준**: 최종 분석 단위는 **행정동 × 분기**로 통일한다. 거래량은 법정동 → 행정동 매핑 후 집계.
- **변화율 계산 기준**: 동일 공간 단위 내 전기(전월) 대비 변화율을 우선 사용한다.
- **이상치 처리 기준**: 단순 절대값 컷이 아니라 분포(IQR) 및 보조 변수(면적-금액)와의 관계를 함께 보고 판단한다.
- **결측치 처리 기준**: 의미가 명확한 결측은 의미 있는 라벨(`whole_building`, `unknown`)로 대체. 의미 불명 결측은 분석에서 제외 또는 별도 표기.

## 7. 전처리 실행 코드 (요약)

원본 노트북(`new_거래량_2차 전처리.ipynb`, `인구_기업_전처리.ipynb`)의 핵심 처리 로직을 한 곳에 모은다.

In [ ]:
# 7-1. 거래량 전처리
trans_df = loaded['transactions'].copy()

# 1) 층 결측치 처리: 유형(일반/집합)에 따라 라벨 부여
trans_df['층'] = trans_df['층'].astype('object')
trans_df.loc[(trans_df['유형'] == '집합') & (trans_df['층'].isna()), '층'] = 'unknown'
trans_df.loc[(trans_df['유형'] == '일반') & (trans_df['층'].isna()), '층'] = 'whole_building'

# 2) 거래금액 이상치 추출 (제거가 아닌 검토용)
lower_10 = np.percentile(trans_df['거래금액(만원)'], 10)
upper_90 = np.percentile(trans_df['거래금액(만원)'], 90)

upper_subset = trans_df[trans_df['거래금액(만원)'] >= upper_90]
q1_u, q3_u = upper_subset['거래금액(만원)'].quantile([0.25, 0.75])
upper_extreme = upper_subset[upper_subset['거래금액(만원)'] > (q3_u + 1.5 * (q3_u - q1_u))]

lower_subset = trans_df[trans_df['거래금액(만원)'] <= lower_10]
q1_l, q3_l = lower_subset['거래금액(만원)'].quantile([0.25, 0.75])
# 하위 1.5·IQR 컷은 0건 → 0.5·IQR로 완화
lower_extreme = lower_subset[lower_subset['거래금액(만원)'] < (q1_l - 0.5 * (q3_l - q1_l))]

print(f'상위 극단값: {len(upper_extreme)}건, 하위 극단값: {len(lower_extreme)}건')

# 3) 계약연월 → datetime, 분기 파생
trans_df['계약연월'] = pd.to_datetime(trans_df['계약연월'], format='%Y-%m')
trans_df['분기'] = trans_df['계약연월'].dt.to_period('Q')

# 4) 산출물: 실거래가_2차_전처리.csv (이미 폴더에 존재)
# trans_df.to_csv(FILE_PATH / '실거래가_2차_전처리.csv', index=False, encoding='utf-8-sig')
display(trans_df.head())

In [ ]:
# 7-2. 인구 전처리
pop_df = loaded['population'].copy()

# 콤마 포함 문자열 → int 변환
pop_df['총인구수'] = pop_df['총인구수'].str.replace(',', '').astype(int)
pop_df['세대수']   = pop_df['세대수'].str.replace(',', '').astype(int)
pop_df.info()

In [ ]:
# 7-3. 기업 전처리: 결측/중복 0건이라 별도 처리 불필요. 그대로 사용.
corp_df = loaded['new_corp'].copy()
print(corp_df.shape)
display(corp_df.head())

## 8. EDA 주요 결과 요약

원본 `EDA.ipynb`의 결과를 카테고리별로 정리.

### 8-1. 거래량 변화 (분기별 법정동 거래율)
- 2023Q1 ~ 2025Q4 총 12분기 동안 분기별 거래량 비율 상위 12개 법정동을 추출.
- **전체 12분기 모두 상위에 든 법정동 (8개)**: (분당구) 서현동, 구미동, 정자동, 야탑동, 삼평동, 수내동, 금곡동 / (수정구) 창곡동
- 그 외: 성남동(중원구) 10회, 상대원동(중원구) 8회, 이매동·운중동·신흥동·백현동·대장동 등.
- 시각화: 8개 법정동의 분기별 거래 비율 변화 라인 차트.

### 8-2. 거래량 도메인 조사 (위험 수준)

| 지역명 | 위험 수준 | 위험 유형 | 핵심 |
|---|---|---|---|
| 삼평동 | 매우 높음 | 업무/상업 복합 | 판교테크노밸리 중심, 거대 자본 위주 상권 재편 |
| 정자동 | 높음 | 상업 젠트리피케이션 | 카페거리 고급화, 소규모 진입 장벽 ↑ |
| 백현동 | 높음 | 상업 젠트리피케이션 | 판교역 인근 카페거리 브랜드화 |
| 서현동 | 보통/주의 | 상업 고착화 | 분당 최대 노후 상권, 업종 교체 주기 가팔라짐 |
| 수내동 | 보통 | 주거/상업 안정 | 학원가 연계 업종 임대료 경쟁 |
| 야탑동 | 보통 | 교통 요충지 | 터미널·병원 유동인구로 임대료 견고 |
| 창곡동 | 안정/유지 | 위례신도시 | 초기부터 높은 임대료, 조정기 |
| 금곡동 | 낮음/안정 | 주거 중심 | 미금역 인근 외 안정적 |
| 구미동 | 낮음/안정 | 주거 중심 | 분당 남단 성숙 주거지 |

→ 거래량과 연관성이 예상되는 변수: **공시지가 상승률(임대료), 프랜차이즈 침투율, 업종 교체 주기, 유동인구, 공실률**

### 8-3. 구별 용도지역 / 건축물주용도 분포
- 분당구·수정구·중원구 3개 구의 용도지역(중심상업/준주거/일반상업/근린상업/주거지역 등) 분포를 막대그래프로 비교.
- 분당구·수정구·중원구의 건축물주용도(근린생활/업무/판매/교육연구/숙박/기타) 분포 비교.

### 8-4. 신규 기업 증가율
- `new_corp_total.csv`를 `date × admi_nm` 피벗 후 `pct_change()`로 행정동별 신규 법인 수 월별 증가율 산출.
- 누적 신규 법인 수 상위 5개 행정동의 증가율 추이 라인 차트.

### 8-5. 매출 변화율 (카드 매출)
- **지역별(구) 총매출액 / 총거래건수**: 분당구(신도심) 41135, 수정구(원도심) 41133, 중원구(원도심) 41131 비교.
- **월별 매출액 / 거래건수 추이**: 3개 구 라인 차트, 매출액(십억원), 거래건수(만건) 단위.
- **2025년 구별 행정동 매출 Top 5**
  - 분당구: 정자1동 65.1조원 / 백현동 17.0조원 / 정자3동 7.7조원 / 서현1동 6.1조원 / 수내1동 6.0조원
  - 중원구: 상대원1동 3.0조원 / 도촌동 2.0조원 / 성남동 0.40조원 / 하대원동 0.16조원 / 금광2동 0.11조원
  - 수정구: 위례동 0.49조원 / 태평4동 0.22조원 / 수진2동 0.17조원 / 신흥3동 0.15조원 / 시흥동 0.14조원
- **원도심(중원·수정) 행정동 Top 10**: 상대원1동, 도촌동, 위례동, 성남동, 태평4동, 수진2동, 하대원동, 신흥3동, 시흥동, 신흥2동.

### 8-6. 매출 도메인 조사 (위험 수준 종합)

| 구 | 매우 높음 | 높음 | 보통/주의 | 낮음/안정 |
|---|---|---|---|---|
| 분당구 | 백현동 | 정자1동 | 서현1동 | 수내1동, 정자3동 |
| 중원구 | 성남동 | 금광2동 | 상대원1동 | 하대원동, 도촌동 |
| 수정구 | 신흥3동 | 수진2동, 태평4동 | 성남동(수정구) | 위례동(안정), 시흥동(특수: 토지 가격 급등) |

**핵심 인사이트**: 분당 지역은 매출 절대 규모가 매우 크므로, **객단가(amt/cnt)** 변화율로 보면 젠트리피케이션 속도를 더 명확히 볼 수 있음.

## 9. 전처리 결과 요약표 (데이터셋별)

### 거래량
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| transactions_total.csv | 실거래가_2차_전처리.csv | 2023-01 ~ 2025-12 | 3,060 | 시군구, 법정동, 계약연월, 거래금액(만원), 전용/연면적(㎡), 용도지역, 건축물주용도, 층 | 투자/거래 활동 파악 |

### 인구
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| population_total.csv | (변환 후 동일) | 2023-01 ~ 2025-12 | 1,800 | 행정구역, 행정동 코드, 날짜, 총인구수, 세대수, 세대당_인구 | 배후 수요 파악 |

### 기업
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| new_corp_total.csv | (그대로 사용) | 2023-01 ~ 2025-12 | 3,367 | date, admi_nm, dong_code, induty_pri_nm, induty_med_nm, ncr_crp_comp_cn | 상권 변화 신호 파악 |

### 매출
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 전처리완료_카드(매출).csv | (그대로 사용, EDA 단계 집계) | 2023-01 ~ 2025-12 | (대용량) | ta_ymd, cty_rgn_no, admi_cty_no, card_tpbuz_*, amt, cnt | 소비 활동 / 객단가 변화 |

## 10. 수준 변수 / 변화율 변수 구분표 + 최종 변수 연결표

### 수준 변수 / 변화율 변수
| 구분 | 변수명 | 의미 | 해석 |
|---|---|---|---|
| 수준 | transaction_cnt | 행정동·월 거래 건수 | 투자/거래 활동 수준 |
| 수준 | transaction_amt | 행정동·월 평균/합계 거래금액 | 부동산 가치 수준 |
| 수준 | population_total | 총인구 | 배후 수요 규모 |
| 수준 | household_cnt | 세대수 | 배후 수요 단위 수 |
| 수준 | new_corp_cnt | 신규 법인 수 | 상권 신규 진입 활동 |
| 수준 | sales_amt | 카드 매출액 | 소비 활동 수준 |
| 수준 | sales_per_cnt (객단가) | amt / cnt | 단가 수준 (젠트리피케이션 속도 핵심) |
| 변화율 | transaction_cnt_growth | 거래 건수 전월 대비 변화율 | 거래 활성화/위축 신호 |
| 변화율 | population_total_growth | 인구 변화율 | 인구 유출입 |
| 변화율 | new_corp_cnt_growth | 신규 법인 변화율 | 상권 변화 속도 |
| 변화율 | sales_per_cnt_growth | 객단가 변화율 | 가격 상승 압력 |

### 최종 변수 연결표
| 데이터셋 | 원본 컬럼 | 최종 변수명 | 의미 | 사용 방향 |
|---|---|---|---|---|
| 거래량 | (집계) 행 수 | transaction_cnt | 거래 활동 수준 | 투자 유입 압력 |
| 거래량 | 거래금액(만원) | transaction_amt | 거래 금액 수준 | 부동산 가치 수준 |
| 거래량 | 용도지역 | zone_type | 용도지역 코드 | 상업화 단계 해석 |
| 거래량 | 건축물주용도 | bldg_use | 건축물 용도 | 업종 변화 해석 |
| 인구 | 총인구수 | population_total | 인구 | 배후 수요 |
| 인구 | 세대수 | household_cnt | 세대수 | 가구 단위 수요 |
| 인구 | 세대당_인구 | persons_per_household | 가구당 인구 | 가구 구성 변화 |
| 기업 | ncr_crp_comp_cn | new_corp_cnt | 신규 법인 수 | 상권 신규 진입 |
| 기업 | induty_pri_nm | industry_major | 산업 대분류 | 업종 변화 |
| 매출 | amt | sales_amt | 매출액 | 소비 활동 |
| 매출 | cnt | sales_cnt | 거래 건수 | 소비 빈도 |
| 매출 | amt/cnt | sales_per_cnt | 객단가 | 가격 상승 압력 |

### 젠트리피케이션 해석 칸
- **투자 유입 압력**: transaction_cnt, transaction_cnt_growth (특히 분당 8개 동·창곡동·성남동 모니터링)
- **부동산 가치 수준**: transaction_amt (용도지역·건축물주용도 통제 후 비교)
- **가격 상승 압력**: sales_per_cnt_growth (분당 백현동·정자1동 우선)
- **배후 수요**: population_total, household_cnt (재개발 진행 중인 신흥3동·수진2동·태평4동 변화 주목)
- **상권 변화 신호**: new_corp_cnt_growth, 산업 대분류별 신규 법인 비중 변화

## 11. 최종 체크리스트 + 마지막 요약

### 체크리스트
- [x] 최종본 / 백업 / 삭제예정 파일 구분 완료
- [x] 거래량 전처리 결과 정리 완료 (층 결측치, 이상치 추출)
- [x] 인구 전처리 결과 정리 완료 (타입 변환)
- [x] 기업 데이터 점검 완료 (결측·중복 0)
- [x] 매출 EDA 결과 정리 완료 (지역/월/행정동/원도심 Top10)
- [x] 시간 단위(월) / 공간 단위(행정동) 확정 완료
- [x] 수준 변수 / 변화율 변수 구분표 작성 완료
- [x] 최종 변수 연결표 작성 완료
- [x] 젠트리피케이션 해석 칸 작성 완료
- [ ] 거래량 면적 이상치 컷 기준 확정 (검토 중)
- [ ] 거래량 법정동 ↔ 행정동 매핑 확정
- [ ] 거래량·인구·기업·매출 행정동·월 단위 통합 marts 파일 저장

### 팀 공유용 5줄 요약
1. 성남시 분당·수정·중원 3개 구의 부동산 거래량(3,060건) / 인구(1,800행) / 신규 기업(3,367행) / 카드 매출 데이터를 행정동·월 기준으로 정리했습니다.
2. 거래량의 핵심 결측은 `층`(46%)이며 유형(일반/집합)에 따라 `whole_building` / `unknown`으로 라벨링했고, 거래금액·면적 이상치는 IQR 기반으로 추출해 별도 검토 중입니다.
3. 분기별 거래량 상위 동(분당구 7개 + 수정구 창곡동) 8곳을 식별했고, 도메인 조사 결과 삼평·정자·백현이 매우 높음/높음 위험으로 분류됩니다.
4. 매출 EDA에서는 분당 백현동·정자1동, 수정구 신흥3동·수진2동·태평4동, 중원구 성남동·금광2동이 젠트리피케이션 위험 핵심지로 확인됐습니다.
5. 다음 단계는 거래량 면적 이상치 컷 확정, 법정동↔행정동 매핑, 그리고 4개 데이터셋의 행정동·월 단위 통합 marts 파일 산출입니다.